# Wikipedia Q&A — Prototype

A system that answers questions using **Claude Sonnet 4.6** + a `search_wikipedia` tool,
plus an **eval suite** that grades it. The model decides when to search, can search multiple
times to refine, and grounds its answer in what it retrieves.

## Setup
1. `cp .env.example .env` and add your `ANTHROPIC_API_KEY`
2. Select the `.venv` kernel (top-right)
3. Run the cells top to bottom

The Wikipedia layer (section 1) needs **no API key** — you can test retrieval first.

In [84]:
import os
import json
import time
import requests
import anthropic
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env

MODEL = "claude-sonnet-4-6"          # the agent that reasons + dispatches searches
client = anthropic.Anthropic()        # picks up ANTHROPIC_API_KEY from the environment
print("anthropic", anthropic.__version__, "| key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

anthropic 0.112.0 | key loaded: True


## 1. Wikipedia retrieval

Uses the live **MediaWiki API** (no key, no setup). Two calls: a full-text search for
relevant titles, then plain-text intro extracts for those titles. Returns several results
so the model can pick the relevant one or re-search — Wikipedia's top hit isn't always the
best match.

In [85]:
import threading

WIKI_API = "https://en.wikipedia.org/w/api.php"
# Descriptive User-Agent (Wikipedia policy throttles generic ones). Contact is a placeholder.
WIKI_HEADERS = {"User-Agent": "WikiQA-Eval/1.0 (take-home eval; https://github.com/example/wikiqa-eval)"}

NO_RESULTS_MSG = "No Wikipedia articles found"
SEARCH_FAILED_MSG = "[search_wikipedia failed"

# --- Persistent cache -------------------------------------------------------------
# The biggest lever against 429s is sending FEWER requests. We cache successful results to
# disk: repeated queries (within a run, across retries, and across re-runs) never re-hit
# Wikipedia. Failures are NOT cached, so a 429'd query is retried fresh next time.
_WIKI_CACHE_PATH = "eval_results/wiki_cache.json"
_WIKI_CACHE_LOCK = threading.Lock()
_wiki_cache = None


def _cache():
    global _wiki_cache
    if _wiki_cache is None:
        _wiki_cache = json.load(open(_WIKI_CACHE_PATH)) if os.path.exists(_WIKI_CACHE_PATH) else {}
    return _wiki_cache


def _cache_get(key):
    with _WIKI_CACHE_LOCK:
        return _cache().get(key)


def _cache_put(key, value):
    with _WIKI_CACHE_LOCK:
        _cache()[key] = value
        os.makedirs("eval_results", exist_ok=True)
        with open(_WIKI_CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(_wiki_cache, f, ensure_ascii=False)


# --- Global rate-limiter ----------------------------------------------------------
_WIKI_LOCK = threading.Lock()
_WIKI_MIN_INTERVAL = 1.0   # seconds between request starts, globally (~1 req/s — conservative)
_wiki_last = [0.0]


def _wiki_throttle():
    with _WIKI_LOCK:
        wait = _WIKI_MIN_INTERVAL - (time.perf_counter() - _wiki_last[0])
        if wait > 0:
            time.sleep(wait)
        _wiki_last[0] = time.perf_counter()


def _wiki_get(params: dict, retries: int = 4) -> dict:
    """GET the MediaWiki API: globally throttled, retries with exponential backoff, honors
    Retry-After on 429."""
    last_err = None
    for attempt in range(retries + 1):
        _wiki_throttle()
        try:
            r = requests.get(WIKI_API, params=params, headers=WIKI_HEADERS, timeout=60)
            if r.status_code == 429:
                wait = float(r.headers.get("Retry-After", 0)) or (0.5 * 2 ** attempt)
                last_err = requests.HTTPError("429 Too Many Requests")
                time.sleep(min(wait, 30))
                continue
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as e:
            last_err = e
            time.sleep(0.5 * (2 ** attempt))  # exponential backoff
    raise last_err


def search_wikipedia(query: str, max_results: int = 3, chars_per_article: int = 1500) -> str:
    """Search Wikipedia, return intro extracts from the top matches. Cached; never raises
    (returns a sentinel string on failure)."""
    key = f"{query}|{max_results}|{chars_per_article}"
    hit = _cache_get(key)
    if hit is not None:
        return hit
    try:
        search = _wiki_get({
            "action": "query", "list": "search", "srsearch": query,
            "srlimit": max_results, "format": "json",
        })
        hits = search.get("query", {}).get("search", [])
        if not hits:
            result = f"{NO_RESULTS_MSG} for query: {query!r}"
            _cache_put(key, result)
            return result
        titles = [h["title"] for h in hits]

        extracts = _wiki_get({
            "action": "query", "prop": "extracts|info",
            "exintro": True, "explaintext": True, "inprop": "url",
            "titles": "|".join(titles), "format": "json",
        })
        by_title = {p["title"]: p for p in extracts.get("query", {}).get("pages", {}).values()}

        blocks = []
        for title in titles:  # keep search-rank order
            page = by_title.get(title, {})
            extract = (page.get("extract") or "").strip()
            if len(extract) > chars_per_article:
                extract = extract[:chars_per_article].rsplit(" ", 1)[0] + "…"
            url = page.get("fullurl", "")
            blocks.append(f"## {title}\n{url}\n\n{extract or '(no extract available)'}")
        result = "\n\n---\n\n".join(blocks)
        _cache_put(key, result)   # cache only successful retrievals
        return result
    except Exception as e:
        return (f"{SEARCH_FAILED_MSG} for {query!r}: {type(e).__name__}: {e}. "
                f"Wikipedia may be temporarily unavailable — retry once, or tell the user you "
                f"could not retrieve sources.]")

In [ ]:
# No API key needed — sanity-check the retrieval layer
print(search_wikipedia("Who painted the Mona Lisa?", max_results=2, chars_per_article=400))

## 2. Tool definition + system prompt

These are the heart of the assignment. Kept inline while we iterate; once stable they move to
`prompts.py`.

In [86]:
TOOLS = [
    {
        "name": "search_wikipedia",
        "description": (
            "Search English Wikipedia and return intro extracts from the top matching "
            "articles. Use this to ground answers in real sources. You may call it multiple "
            "times with refined queries if the first results aren't relevant. The top result "
            "is not always the best match \u2014 read all returned articles before deciding."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search terms. Prefer concise topic phrases (e.g. 'Mount Everest height') over full questions.",
                }
            },
            "required": ["query"],
        },
    }
]

In [95]:
SYSTEM_PROMPT = """\
<Role>

You are a Wikipedia-grounded question answering assistant.

Your goal is to answer the user's question accurately using information retrieved from Wikipedia. Treat the retrieved Wikipedia content as the authoritative source for factual claims.

When sufficient evidence is unavailable, be honest about the limitation rather than guessing.

</Role>

<SearchStrategy>

Decide whether the user's request requires retrieval.

- Use search_wikipedia whenever answering a question that depends on external factual knowledge.
- Do not search for purely creative writing, arithmetic, translation, rewriting, or other tasks that do not require encyclopedic knowledge.

When searching:

- Use short, keyword-based queries rather than full natural-language questions.
- Prefer entity names or specific concepts.
- For multi-part questions, decompose the problem into multiple targeted searches.
- Read all returned articles before deciding whether another search is necessary.
- If the retrieved results are incomplete, ambiguous, or irrelevant, refine the query and search again.
- Continue searching until:
  - the retrieved evidence is sufficient to answer the question,
  - additional searches are unlikely to provide new information, or
  - the search budget has been reached.

</SearchStrategy>

<GroundingPolicy>

Base factual claims only on the retrieved Wikipedia content.

Every factual claim in your answer should be traceable to the retrieved evidence.

Do not introduce facts that are not supported by the retrieved content, even if you believe they are correct from prior knowledge.

If the retrieved evidence is insufficient:

- explicitly say what information is missing,
- avoid speculation,
- do not fabricate names, dates, quotations, statistics, or other facts.

If the user's question contains a false premise, use the retrieved evidence to explain why the premise is incorrect before answering the underlying question when possible.

</GroundingPolicy>

<AnswerPolicy>

Answer the user's question directly before providing supporting detail.

Be concise while remaining complete.

Use only the amount of detail needed to answer the user's question.

When factual information comes from Wikipedia, cite the relevant article(s), for example:

(Source: Mount Everest)

If multiple articles contributed to the answer, cite each one.

If Wikipedia does not contain enough information to answer confidently, clearly state that instead of guessing.

Before producing your final answer, verify that every factual claim is supported by the retrieved evidence.

</AnswerPolicy>
"""

## 3. The agent loop

A manual tool-use loop so we can see **whether search was used**, how many times, with what
queries, and **what text it retrieved** (the last is fed to the judge to grade groundedness).

In [88]:
def _classify_search(result_text: str) -> str:
    """ok | empty | failed, based on the sentinel returned by search_wikipedia."""
    if result_text.startswith(SEARCH_FAILED_MSG):
        return "failed"
    if result_text.startswith(NO_RESULTS_MSG):
        return "empty"
    return "ok"


def answer_question(question: str, max_turns: int = 6, verbose: bool = False) -> dict:
    """Answer a question with Claude + Wikipedia.

    Returns the answer plus a full trace: per-search status, per-TURN structure (so we can
    tell parallel vs sequential multi-hop), and a latency breakdown (Claude vs Wikipedia).
    """
    messages = [{"role": "user", "content": question}]
    searches, search_log = [], []
    turn = 0                      # counts tool-use turns = sequential hops
    t_start = time.perf_counter()
    wiki_seconds = 0.0           # time spent inside search_wikipedia

    for _ in range(max_turns):
        resp = client.messages.create(
            model=MODEL, max_tokens=2048,
            system=SYSTEM_PROMPT, tools=TOOLS, messages=messages,
        )
        if resp.stop_reason != "tool_use":
            break
        turn += 1
        messages.append({"role": "assistant", "content": resp.content})
        tool_results = []
        for block in resp.content:  # a single response may contain MULTIPLE searches (parallel)
            if block.type == "tool_use" and block.name == "search_wikipedia":
                query = block.input["query"]
                searches.append(query)
                if verbose:
                    print(f"  \U0001f50d [turn {turn}] search_wikipedia({query!r})")
                tw = time.perf_counter()
                result = search_wikipedia(query)
                wiki_seconds += time.perf_counter() - tw
                status = _classify_search(result)
                if verbose and status != "ok":
                    print(f"     ! search {status} for {query!r}")
                search_log.append({"turn": turn, "query": query, "result": result, "status": status})
                tool_results.append({
                    "type": "tool_result", "tool_use_id": block.id, "content": result,
                })
        messages.append({"role": "user", "content": tool_results})

    answer = "".join(b.text for b in resp.content if b.type == "text")
    retrieved_text = "\n\n".join(f"[search: {s['query']!r}]\n{s['result']}" for s in search_log)
    search_failures = [{"query": s["query"], "status": s["status"]}
                       for s in search_log if s["status"] != "ok"]
    return {
        "question": question,
        "answer": answer,
        "used_search": len(searches) > 0,
        "num_searches": len(searches),
        "num_turns": turn,                    # tool-use turns; num_searches > num_turns => parallel
        "queries": searches,
        "search_log": search_log,             # each entry tagged with its turn index
        "search_failures": search_failures,
        "retrieved_text": retrieved_text,
        "t_total": time.perf_counter() - t_start,  # wall time of the whole agent loop
        "t_wiki": wiki_seconds,                     # of which, time in Wikipedia calls
    }

## 4. Demo

Needs your `ANTHROPIC_API_KEY` in `.env`.

In [ ]:
result = answer_question("Who painted the Mona Lisa, and where is it displayed today?", verbose=True)
print("\nUsed search:", result["used_search"], "| searches:", result["num_searches"])
print("Queries:", result["queries"])
print("\n" + result["answer"])

## 5. Eval harness

Grades the agent across the dataset in `eval_data.py`. Three quality metrics from an
**LLM judge** (a *stronger* model, Opus 4.8, to reduce self-grading bias) on a 0/1/2 rubric,
plus **deterministic** behavior checks (did it search when expected? did it cite?).

In [89]:
import re
import pandas as pd
from eval_data import EVAL_EXAMPLES, CATEGORIES

JUDGE_MODEL = "claude-opus-4-8"   # different + stronger than the agent -> less self-grading bias
print(f"{len(EVAL_EXAMPLES)} examples across {len(CATEGORIES)} categories: {CATEGORIES}")

23 examples across 7 categories: ['ambiguous', 'false_premise', 'missing_detail', 'multi_hop', 'no_search_needed', 'single_fact', 'unanswerable']


In [90]:
# The rubric IS the judge prompt. Edit here to tune grading.
JUDGE_SYSTEM = """You are a strict, fair grader for a Wikipedia question-answering system.
Score the assistant's answer on four 0/1/2 metrics plus a hallucination flag.

Each question is tagged SOURCING REQUIRED (yes / no). This determines whether the system is
allowed to use its own world knowledge:

- SOURCING REQUIRED = yes  (factual lookups answerable from Wikipedia):
  The system must answer ONLY from the RETRIEVED CONTENT shown — never from its own world
  knowledge. A factual claim not present in the retrieved content is a grounding violation, even
  if correct or well-known: award NO correctness/completeness credit for it, and penalize
  groundedness. If the searches FAILED or returned nothing, the system has no sources — the only
  acceptable answer is a clean refusal that asserts no facts (correctness 0 since it doesn't
  answer, groundedness 2 since it invents nothing). If the answer asserts ANY fact in that
  situation — even a correct one like "Tokyo, the capital of Japan" — score correctness 0 AND
  groundedness 0.

- SOURCING REQUIRED = no  (no lookup needed — arithmetic, translation, creative writing — or
  not answerable from Wikipedia at all):
  The model's own knowledge and judgment are appropriate. Grade normally; do NOT penalize it for
  using general knowledge or for not citing a source. Groundedness 2 unless it states something
  it presents as a retrieved fact that is actually wrong.

CORRECTNESS (vs the REFERENCE; for SOURCING REQUIRED=yes, count only GROUNDED facts):
  2 = key facts match the reference (and, if sourcing required, are supported by retrieved content)
  1 = partly right, or right with a material error
  0 = wrong, contradicts the reference, OR (sourcing required) correct only via ungrounded knowledge
COMPLETENESS (vs the QUESTION):
  2 = fully addresses every part
  1 = answers the core but omits a sub-part or important qualifier
  0 = misses the main thing asked (including a refusal that never provides the answer)
CONCISENESS (relevance, NOT length):
  2 = no superfluous content; every sentence serves the question
  1 = mostly focused, but includes some unrequested detail
  0 = padded, or buried in tangential / ungrounded content
  Judge by relevance to what was asked, never by raw word count.
GROUNDEDNESS (provenance — HOW MUCH of the answer traces to the sources):
  For SOURCING REQUIRED=no, groundedness is 2 (nothing needed grounding) unless a presented-as-fact
  claim is wrong. For SOURCING REQUIRED=yes:
  2 = every factual claim is supported by retrieved content; a clean refusal that asserts no facts
      (when sources are absent/failed) scores 2
  1 = core claims supported, but some unsupported or embellished detail
  0 = asserts facts not in the retrieved content — fabrication, embellishment from memory, or any
      world-knowledge fact stated when the searches failed

HALLUCINATED (true / false) — the key safety signal, scored SEPARATELY from groundedness:
  true  = the answer FABRICATES content that does not exist in the sources: a factual claim that is
          FALSE, an INVENTED quotation, or a statement ATTRIBUTED to Wikipedia / the sources that
          they do not actually make.
  false = no fabricated content.
  Crucial distinction: a TRUE fact recalled from the model's own knowledge, stated WITHOUT a
  fabricated source, is ungrounded (it lowers groundedness) but is NOT a hallucination. Hallucination
  is specifically about MADE-UP content or MADE-UP sources, regardless of whether an underlying fact
  happens to be true. A clean refusal is never a hallucination.

Be willing to give 0s. Return only the structured verdict."""

JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "correctness": {"type": "integer", "enum": [0, 1, 2]},
        "completeness": {"type": "integer", "enum": [0, 1, 2]},
        "conciseness": {"type": "integer", "enum": [0, 1, 2]},
        "groundedness": {"type": "integer", "enum": [0, 1, 2]},
        "hallucinated": {"type": "boolean"},
        "reasoning": {"type": "string", "description": "1-2 sentences justifying the scores."},
    },
    "required": ["correctness", "completeness", "conciseness", "groundedness", "hallucinated",
                 "reasoning"],
    "additionalProperties": False,
}


def judge_answer(example: dict, answer: str, retrieved_text: str) -> dict:
    """Grade one answer with the judge model. Returns the rubric verdict as a dict."""
    sourcing = "yes" if example.get("expected_search") else "no"
    user = (
        f"SOURCING REQUIRED: {sourcing}\n\n"
        f"QUESTION:\n{example['question']}\n\n"
        f"REFERENCE (gold answer / key facts):\n{example['reference']}\n\n"
        f"RETRIEVED CONTENT (what the assistant saw from Wikipedia):\n"
        f"{retrieved_text or '(the assistant did not search)'}\n\n"
        f"ASSISTANT'S ANSWER:\n{answer}\n\nScore the answer."
    )
    resp = client.messages.create(
        model=JUDGE_MODEL, max_tokens=1024, system=JUDGE_SYSTEM,
        output_config={"format": {"type": "json_schema", "schema": JUDGE_SCHEMA}},
        messages=[{"role": "user", "content": user}],
    )
    return json.loads(next(b.text for b in resp.content if b.type == "text"))

In [91]:
def behavior_checks(example: dict, result: dict) -> dict:
    """Deterministic, no-API checks from the agent trace."""
    queries = result.get("queries") or []
    # grounded_chain: for dependent multi-hops, the intermediate fact must be DISCOVERED via
    # search, not assumed from memory. We catch the shortcut by checking whether the FIRST query
    # already contains the entity the model was supposed to look up (e.g. "Tokyo" for "capital of
    # Japan"). N/A (None) for questions without a must_discover hop. num_turns can't catch this —
    # a model can inject the answer into query #1 yet still take several turns.
    must = [t.lower() for t in (example.get("must_discover") or [])]
    if must:
        first_q = queries[0].lower() if queries else ""
        grounded_chain = bool(queries) and not any(t in first_q for t in must)
    else:
        grounded_chain = None
    return {
        # did the agent search exactly when we expected it to?
        "search_match": result["used_search"] == example["expected_search"],
        # did the answer include a citation marker? (cheap heuristic)
        "cited": bool(re.search(r"source|wikipedia\.org", result["answer"], re.I)),
        # did it look up the intermediate fact rather than assume it? (None = N/A)
        "grounded_chain": grounded_chain,
    }

In [92]:
from concurrent.futures import ThreadPoolExecutor, as_completed

TRACE_TRUNC = 600   # chars of each Wikipedia result kept in the trace log


def _num(x):
    """Coerce JSON null -> NaN so metric columns stay numeric."""
    return float("nan") if x is None else x


def _trunc_rec(rec: dict) -> dict:
    """Copy of a record with each Wikipedia result truncated for the on-disk log."""
    out = dict(rec)
    out["search_log"] = [
        {"turn": s.get("turn"), "query": s.get("query"), "status": s.get("status"),
         "result": (s["result"][:TRACE_TRUNC] + f"…[+{len(s['result']) - TRACE_TRUNC} chars]")
                   if len(s.get("result", "")) > TRACE_TRUNC else s.get("result", "")}
        for s in rec.get("search_log", [])
    ]
    return out


def _flatten_row(rec: dict) -> dict:
    """Metric row (CSV / DataFrame) from a full record. Missing values -> NaN."""
    j = rec.get("judge") or {}
    tm = rec.get("timing") or {}
    return {
        "id": rec["id"], "category": rec["category"],
        "correctness": _num(j.get("correctness")), "completeness": _num(j.get("completeness")),
        "conciseness": _num(j.get("conciseness")), "groundedness": _num(j.get("groundedness")),
        "hallucinated": _num(j.get("hallucinated")),   # bool -> 1/0 in aggregates; NaN if errored
        "search_match": _num(rec.get("search_match")), "cited": _num(rec.get("cited")),
        "grounded_chain": _num(rec.get("grounded_chain")),  # True/False for must_discover hops; else NaN
        "num_searches": _num(rec.get("num_searches")), "num_turns": _num(rec.get("num_turns")),
        "queries": " | ".join(rec.get("queries") or []),   # the actual Wikipedia search queries
        "n_search_failures": len(rec.get("search_failures") or []),
        "t_agent_llm": _num(tm.get("agent_llm")), "t_wiki": _num(tm.get("wiki")),
        "t_judge": _num(tm.get("judge")),
        "expected_search": rec.get("expected_search"), "used_search": _num(rec.get("used_search")),
        "error": rec.get("error"),
        "answer": rec.get("answer", ""), "judge_reasoning": j.get("reasoning", ""),
    }


def _eval_one(ex: dict) -> dict:
    """Run agent + judge + checks for one example; return a full record (never raises)."""
    try:
        result = answer_question(ex["question"])
        t_j = time.perf_counter()
        verdict = judge_answer(ex, result["answer"], result["retrieved_text"])
        t_judge = time.perf_counter() - t_j
        checks = behavior_checks(ex, result)
        return {
            "id": ex["id"], "category": ex["category"], "question": ex["question"],
            "expected_search": ex["expected_search"], "reference": ex["reference"],
            "used_search": result["used_search"], "num_searches": result["num_searches"],
            "num_turns": result["num_turns"],
            "queries": result["queries"], "search_log": result["search_log"],
            "search_failures": result["search_failures"],
            "answer": result["answer"], "judge": verdict,
            "search_match": checks["search_match"], "cited": checks["cited"],
            "grounded_chain": checks["grounded_chain"],
            "timing": {"agent_total": round(result["t_total"], 2),
                       "agent_llm": round(result["t_total"] - result["t_wiki"], 2),
                       "wiki": round(result["t_wiki"], 2), "judge": round(t_judge, 2)},
            "error": None,
        }
    except Exception as e:
        return {
            "id": ex["id"], "category": ex["category"], "question": ex["question"],
            "expected_search": ex["expected_search"], "reference": ex.get("reference", ""),
            "used_search": None, "num_searches": None, "num_turns": None,
            "queries": [], "search_log": [], "search_failures": [],
            "answer": f"[ERROR: {e}]", "judge": None, "timing": None,
            "search_match": None, "cited": None, "grounded_chain": None,
            "error": f"{type(e).__name__}: {e}",
        }


def run_evals(examples: list, ts: int = None, resume: bool = True, verbose: bool = True,
              max_workers: int = 3):
    """Run evals, writing each example to disk AS IT COMPLETES (incremental).

    Output is keyed by a single run timestamp `ts` (no SI version). All files for one run share
    that ts: results_{ts}.jsonl here, plus eval_results_/trace_/scorecard_{ts} from the save fns.
    Pass the same `ts` from run-full so every file lines up. `resume=True` resumes the SAME
    ts log if it exists (crash recovery); a fresh ts starts clean. Returns (metrics_df, records).
    """
    ts = int(time.time()) if ts is None else ts
    os.makedirs("eval_results", exist_ok=True)
    log_path = f"eval_results/results_{ts}.jsonl"

    if not resume and os.path.exists(log_path):
        os.remove(log_path)

    done_ok = set()
    if os.path.exists(log_path):
        with open(log_path, encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                if not r.get("error"):
                    done_ok.add(r["id"])
        if done_ok:
            print(f"resume: {len(done_ok)} example(s) already complete -> skipping\n")

    todo = [e for e in examples if e["id"] not in done_ok]

    def _emit(rec, logf):  # write-through + print; called only from the main thread
        logf.write(json.dumps(_trunc_rec(rec), ensure_ascii=False) + "\n")
        logf.flush()
        if not verbose:
            return
        if rec["error"]:
            print(f"  {rec['id']:18s} ERROR: {rec['error']}  (recorded, run continues)")
        else:
            j, nf = rec["judge"], len(rec["search_failures"])
            flag = f"  search_fails={nf}" if nf else ""
            hall = "  HALLUC" if j.get("hallucinated") else ""
            chain = "  CHAIN-SHORTCUT" if rec.get("grounded_chain") is False else ""
            print(f"  {rec['id']:18s} C{j['correctness']} Cmp{j['completeness']} "
                  f"Con{j['conciseness']} G{j['groundedness']} "
                  f"search_ok={rec['search_match']}{flag}{hall}{chain}")

    with open(log_path, "a", encoding="utf-8") as logf:
        if max_workers and max_workers > 1 and len(todo) > 1:
            with ThreadPoolExecutor(max_workers=max_workers) as pool:
                futures = [pool.submit(_eval_one, ex) for ex in todo]
                for fut in as_completed(futures):
                    _emit(fut.result(), logf)
        else:
            for ex in todo:
                _emit(_eval_one(ex), logf)
    return load_results(ts)


def load_results(ts: int):
    """Read results_{ts}.jsonl, dedupe by id (last wins), return (metrics_df, records)."""
    log_path = f"eval_results/results_{ts}.jsonl"
    by_id = {}
    with open(log_path, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            by_id[r["id"]] = r  # last occurrence wins (re-runs overwrite earlier attempts)
    records = list(by_id.values())
    df = pd.DataFrame([_flatten_row(r) for r in records])
    return df, records


def summarize(df: pd.DataFrame) -> None:
    metrics = ["correctness", "completeness", "conciseness", "groundedness"]
    n_err = int(df["correctness"].isna().sum())
    if n_err:
        print(f"(!) {n_err} example(s) errored (NaN) and are excluded from the means below\n")
    print("=== Overall means (0-2) ===")
    print(df[metrics].mean().round(2).to_string())
    n_hall = int(df["hallucinated"].astype(float).sum()) if "hallucinated" in df.columns else 0
    print(f"hallucinations (count):   {n_hall}   <- should be 0")
    if "grounded_chain" in df.columns and df["grounded_chain"].notna().any():
        gc = df["grounded_chain"].astype(float)
        print(f"grounded_chain (rate):    {gc.mean():.2f}  ({int(gc.sum())}/{int(gc.notna().sum())} "
              f"dependent hops looked up, not assumed)")
    print(f"tool-use accuracy:        {df['search_match'].astype(float).mean():.2f}")
    print(f"citation rate:            {df['cited'].astype(float).mean():.2f}")
    print(f"avg # searches:           {df['num_searches'].mean():.2f}")
    print(f"search failures (total):  {int(df['n_search_failures'].sum())}")
    if {"t_agent_llm", "t_wiki", "t_judge"}.issubset(df.columns):
        tm = df[["t_agent_llm", "t_wiki", "t_judge"]].mean()
        print("\n=== Avg latency / example (seconds) ===")
        print(f"agent (Claude calls): {tm['t_agent_llm']:.1f}")
        print(f"wikipedia:            {tm['t_wiki']:.1f}")
        print(f"judge (Opus call):    {tm['t_judge']:.1f}")
        print(f"~ total / example:    {tm.sum():.1f}")


def save_results(df: pd.DataFrame, ts: int = None) -> str:
    """Snapshot the per-example metrics (incl. the search queries) as a timestamped CSV."""
    ts = int(time.time()) if ts is None else ts
    os.makedirs("eval_results", exist_ok=True)
    path = f"eval_results/eval_results_{ts}.csv"
    df.to_csv(path, index=False)
    print(f"saved metrics -> {path}")
    return path


def save_trace(records: list, ts: int = None) -> str:
    """Snapshot the full traces as a pretty-printed, timestamped JSON array."""
    ts = int(time.time()) if ts is None else ts
    os.makedirs("eval_results", exist_ok=True)
    path = f"eval_results/trace_{ts}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"saved traces  -> {path}  (pretty — open in VS Code)")
    return path

In [93]:
CITE_EXEMPT = {"missing_detail"}      # absence-reporting answers correctly cite nothing
GROUND_EXEMPT = {"no_search_needed"}  # no factual claims to ground -> groundedness is N/A


def scorecard(df: pd.DataFrame, ts: int = None, write: bool = True) -> pd.DataFrame:
    """Consolidated, hill-climbable metrics by category + ALL, in two sections:
    Quality (0-2) and System behavior. Output is keyed by run timestamp `ts`.

    Conditioned metrics (so non-applicable examples don't distort them):
      - `citation*`        : cited rate among examples expected to give a sourced answer
                             (expected_search=True, excluding CITE_EXEMPT absence categories).
      - `groundedness`     : excludes GROUND_EXEMPT (no_search_needed) — a haiku / "17x23" has no
                             factual claims to ground, so it would otherwise inflate the metric.
      - `grounded_chain*`  : among dependent multi-hops only (those with a `must_discover` hop) —
                             rate that the intermediate fact was looked up, not assumed (the first
                             query didn't already contain it). NaN where no such example exists.
    `hallucinations` is a count of answers the judge flagged as fabricating content (target 0).
    Writes scorecard_{ts}.md and appends the ALL row to scorecard_history.csv (keyed by ts).
    """
    ts = int(time.time()) if ts is None else ts
    q = ["correctness", "completeness", "conciseness", "groundedness"]
    work = df.copy()
    for col, default in [("num_turns", float("nan")), ("n_search_failures", 0),
                         ("hallucinated", float("nan")), ("grounded_chain", float("nan"))]:
        if col not in work.columns:   # tolerate runs that predate this instrumentation
            work[col] = default
    for c in ["search_match", "cited"]:
        work[c] = work[c].astype(float)

    cats = sorted(work["category"].dropna().unique())
    rows = []
    for cat in cats + ["ALL"]:
        sub = work if cat == "ALL" else work[work["category"] == cat]
        cite = sub[(sub["expected_search"] == True) & (~sub["category"].isin(CITE_EXEMPT))]
        grnd = sub[~sub["category"].isin(GROUND_EXEMPT)]
        chain = sub["grounded_chain"].astype(float)   # NaN where must_discover absent -> skipped by mean
        rows.append({
            "category": cat, "n": len(sub),
            "correctness": sub["correctness"].mean(), "completeness": sub["completeness"].mean(),
            "conciseness": sub["conciseness"].mean(),
            "groundedness": grnd["groundedness"].mean() if len(grnd) else float("nan"),
            "tool_use_accuracy": sub["search_match"].mean(),
            "citation*": cite["cited"].mean() if len(cite) else float("nan"),
            "grounded_chain*": chain.mean() if chain.notna().any() else float("nan"),
            "avg_searches": sub["num_searches"].mean(), "avg_turns": sub["num_turns"].mean(),
            "search_fails": int(sub["n_search_failures"].sum()),
            "hallucinations": int(sub["hallucinated"].astype(float).sum()),
        })
    sc = pd.DataFrame(rows).set_index("category").round(2)

    def _cell(v):
        if pd.isna(v):
            return "—"
        f = float(v)
        return str(int(f)) if f.is_integer() else f"{f:.2f}"

    def _md(t):
        cols = list(t.columns)
        out = ["| category | " + " | ".join(cols) + " |", "|" + "---|" * (len(cols) + 1)]
        for idx, r in t.iterrows():
            out.append("| " + str(idx) + " | " + " | ".join(_cell(r[c]) for c in cols) + " |")
        return "\n".join(out)

    qual = sc[["n"] + q]
    beh = sc[["n", "tool_use_accuracy", "citation*", "grounded_chain*", "avg_searches",
              "avg_turns", "search_fails", "hallucinations"]]
    text = (f"# Scorecard — {ts}\n\n"
            f"`groundedness` excludes no_search_needed (nothing to ground). "
            f"`citation*` = cited rate among examples expected to give a sourced answer "
            f"(searched, excluding absence-reporting categories). "
            f"`grounded_chain*` = among dependent multi-hops, rate the intermediate fact was looked "
            f"up not assumed. `search_fails` / `hallucinations` are counts — lower is better, "
            f"target 0.\n\n"
            f"## Quality (0–2)\n\n{_md(qual)}\n\n"
            f"## System behavior\n\n{_md(beh)}\n")
    print(text)

    if write:
        os.makedirs("eval_results", exist_ok=True)
        path = f"eval_results/scorecard_{ts}.md"
        with open(path, "w", encoding="utf-8") as f:
            f.write(text)
        all_row = {"ts": ts,
                   **{k: sc.loc["ALL", k] for k in q +
                      ["tool_use_accuracy", "citation*", "grounded_chain*", "avg_searches",
                       "avg_turns", "search_fails", "hallucinations"]}}
        hist = "eval_results/scorecard_history.csv"
        pd.DataFrame([all_row]).to_csv(hist, mode="a", header=not os.path.exists(hist), index=False)
        print(f"saved scorecard -> {path}\nappended ALL row -> {hist}")
    return sc

In [ ]:
# Cheap smoke test of the harness (~3 examples) before the full run (resume=False -> fresh).
sample = [e for e in EVAL_EXAMPLES if e["id"] in
          ("single_fact_01", "missing_detail_02", "no_search_01")]
sample_df, sample_records = run_evals(sample, resume=False)
summarize(sample_df)

In [96]:
# Full run over all 23 examples (multithreaded). All output files for the run share one timestamp.
ts = int(time.time())
baseline_df, records = run_evals(EVAL_EXAMPLES, ts=ts, resume=False, max_workers=3)
summarize(baseline_df)
save_results(baseline_df, ts=ts)   # eval_results/eval_results_{ts}.csv  (per-example metrics + queries)
save_trace(records, ts=ts)         # eval_results/trace_{ts}.json        (full traces)
scorecard(baseline_df, ts=ts)      # eval_results/scorecard_{ts}.md      (+ scorecard_history.csv row)

  single_fact_01     ERROR: BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CcYSyvCr653DtwYfcMmh3'}  (recorded, run continues)
  single_fact_02     ERROR: BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CcYSyvGKnhB2ei8bhV1V4'}  (recorded, run continues)
  single_fact_03     ERROR: BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CcYSyvA7otvBb9hggvYmU'}  (recorded, run continues)
 

,n,correctness,completeness,conciseness,groundedness,tool_use_accuracy,citation*,grounded_chain*,avg_searches,avg_turns,search_fails,hallucinations
category,,,,,,,,,,,,
ambiguous,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
false_premise,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
missing_detail,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
multi_hop,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
no_search_needed,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
single_fact,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
unanswerable,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
ALL,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
